# Prueba técnica Ciencia de datos
Edgardo Cerda


In [ ]:
# Cosas preparatorias
%pip install pandas numpy scikit-learn
%pip install mysql-connector-python
%pip install dotenv
%pip install openai
%pip install requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached openai-3.55.1-py3-none-any.whl.metadata (45 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpx2-2.16.2-py3-none-any.whl.metadata (10.0 kB)
  Using cached jiter-0.19.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.14.4-py3-none-any.whl.metadata (143 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore2-2.16.2-py3-none-any.whl.metadata (28 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.5-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.5-py3-none-any.whl.metadata (2.6 kB)
Using cached openai-3.55.1-py3-none-any.whl (2.5 MB)
Using cached anyio-4.16.0-py3-none-any.whl (134 kB)
Using cached httpx2-2.16.2-py3-none-any.whl (97 kB)
Using cached httpcore2-2.16.2-


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Pregunta 1

In [64]:
import pandas as pd, numpy as np
import mysql.connector
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# Configuración de la conexión a la base de datos
DB_CONFIG = {
    "host": "sql.freedb.tech",
    "port": 3306,
    "user": "u_y8tlWZ",
    "password": "wSkxw3xbhsQg",
    "database": "freedb_N4nnid0a"
}

# Descargar datos de entrenamiento y testeo
connection = mysql.connector.connect(**DB_CONFIG)

train = pd.read_sql_query("SELECT * FROM train", connection)
test = pd.read_sql_query("SELECT * FROM test", connection)


C:\Users\Usuario\AppData\Local\Temp\ipykernel_20788\1896625210.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  train = pd.read_sql_query("SELECT * FROM train", connection)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_20788\1896625210.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test = pd.read_sql_query("SELECT * FROM test", connection)


In [66]:
# Inspeccionar un poco los datos
print("Dimensiones:")
print("train:", train.shape)
print("test :", test.shape)

train.head()

Dimensiones:
train: (35340, 5)
test : (3927, 5)


,id,actividad,oficio,tareas,cod_final
0,0,producción uva exportación,temporera agrícola,mantenimiento de parronales,01
1,1,corporación municipal,profesor mecánico industrial,"preparar clases, enseñar",84
2,4,contratista de rrhh para obras de construcción,trabajador de construcción,veredas,41
3,5,municipalidad de puerto natales,técnico paramedico,"toma de signos vitales, entrega de horas a pac...",84
4,7,guardias seguridad y proteccion,administrador de contrato,administrar dirigir contratar personal,80


In [84]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

text_columns = ["actividad", "oficio", "tareas"]

# Consolidación de las variables textuales
for frame in (train, test):
    frame[text_columns] = frame[text_columns].fillna("")
    frame["_texto"] = frame[text_columns].agg(" ".join, axis=1)

train["_texto"] = train["_texto"].str.strip()
test["_texto"] = test["_texto"].str.strip()

# Eliminación de registros sin texto o sin clase conocida
train = train[
    train["_texto"].str.len() > 0
].dropna(subset=["cod_final"]).copy()

X_train = train["_texto"]
y_train = train["cod_final"].astype(str)
X_test = test["_texto"]

print("\nNúmero de clases:", y_train.nunique())
print("Distribución de clases:")
print(y_train.value_counts().head())

# Representación TF-IDF combinando palabras y caracteres
vectorizer = FeatureUnion([
    (
        "palabras",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "caracteres",
        TfidfVectorizer(
            lowercase=True,
            analyzer="char",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2,
            max_features=50000
        )
    )
])

model = Pipeline([
    ("tfidf", vectorizer),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

test["cod_final_predicho"] = predictions

print("\nPrimeras predicciones:")
print(test[["cod_final_predicho"]].head())

# Evaluación cuando el conjunto test incluye las respuestas reales
if "cod_final" in test.columns and test["cod_final"].notna().any():
    test_evaluado = test.dropna(subset=["cod_final"]).copy()
    y_test = test_evaluado["cod_final"].astype(str)
    y_pred = test_evaluado["cod_final_predicho"]

    print("\n=== Análisis de resultados ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 macro:", f1_score(y_test, y_pred, average="macro"))

    print("\nReporte por clase:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    etiquetas = sorted(set(y_test) | set(y_pred))
    matriz = confusion_matrix(y_test, y_pred, labels=etiquetas)

    print("\nDistribución de predicciones:")
    print(pd.Series(y_pred).value_counts().head())

    confusiones = []
    for i, real in enumerate(etiquetas):
        for j, predicho in enumerate(etiquetas):
            if i != j and matriz[i, j] > 0:
                confusiones.append((matriz[i, j], real, predicho))

    if confusiones:
        print("\nPrincipales confusiones:")
        for cantidad, real, predicho in sorted(confusiones, reverse=True)[:10]:
            print(f"{real} -> {predicho}: {cantidad}")




Número de clases: 79
Distribución de clases:
cod_final
48    5507
01    3629
84    3495
85    3189
86    1682
Name: count, dtype: int64

Primeras predicciones:
  cod_final_predicho
0                 72
1                 43
2                 01
3                 48
4                 85

=== Análisis de resultados ===
Accuracy: 0.84593837535014
F1 macro: 0.7268943680198554

Reporte por clase:
              precision    recall  f1-score   support

          01       0.94      0.93      0.93       403
          02       0.68      0.91      0.78        35
          03       0.88      0.94      0.91        48
          04       1.00      0.91      0.95        81
          06       1.00      1.00      1.00         2
          07       0.75      0.60      0.67         5
          08       0.53      0.80      0.64        10
          09       0.29      0.50      0.37        10
          10       0.89      0.86      0.87       147
          11       0.63      0.80      0.71        15
          

A pesar de ser un modelo simple, los resultados son buenos: Accuracy de .84 y F1 macro de .72.

Una mejora clara es utilizar un modelo preentrenado de transformer, en vez de un modelo de machine learning simple como el utilizado. Seguramente tendrá un rendimiento considerablemente mayor a un costo relativamente bajo. Sin embargo, un modelo simple como el utilizado puede ser beneficioso en ciertos escenarios donde se prioriza mucho la eficiencia del modelo.

# 2. Procesando texto

Se observa que hay una caída progresiva en las métricas de calidad del modelo a medida que pasa el tiempo, pasando desde un accuracy de .938 en testing a .083 en el mes 10, y Macro F1 de .823 a .71. La caída además parece acelerarse. El Macro F1 es consistentemente menor al accuracy lo que puede indicar que las clases están desbalanceadas y la caída se da principalmente en una clase minoritaria. Con 100 registros por mes, sin embargo, las métricas pueden contener algo de ruido, así que se debe concluir con cuidado, pero la tendencia parece ser clara. Si en meses intermedios o en meses siguientes se repite el patrón, la probabilidad de que la calidad real del modelo esté disminuyendo es alta. 

Por lo tanto, mi hipótesis es que los datos que recibe se está pareciendo cada vez menos a los datos con que se entrenó, lo que se denomina data-drift, por cambios en el comportamiento de los datos de origen. Con pocos meses de testeo no es claro que sea una tendencia permanente, pero es una hipótesis testeable. 

Como estrategia de mitigación, seguiría los siguientes pasos:

1. Verificar que no se haya generado algún cambio en el pipeline del modelo que esté afectando la calidad de las predicciones. Quizás, el cambio se debió a un error humano que se debería descartar. En ese caso, revertir los cambios realizados que pueden haber afectado la calidad del modelo, y agregar validaciones al pipeline de producción.
2. Identificar las clases donde el modelo tiene menor precision o recall, o donde se ha observado el mayor cambio en el tiempo. Esto implica calcular las métricas por clase (precision, recall y F1 de cada una) para saber qué clases se deterioran más.
3. Aplicar medidas que permitan corregir el data-drift del modelo, que se deberían probar para definir cuál es la más efectiva y estable. Entre otras, se sugiere:
  - Volver a etiquetar una submuestra considerando active learning, es decir, sobre aquellas clases y casos con peores métricas de calidad y peor confianza del modelo, y reentranar con dichos datos.
  - Reentrenar el modelo utilizando datos recientes, usando una ventana móvil o dando más peso a datos nuevos.  
  - Aplicar alguna téncica fine-tuning (full, LoRa, QLora u otro) del modelo con datos nuevos de manera periódica. 



# 3. Procesando texto


In [14]:
# Descarga directa de la tabla noticias
noticias = pd.read_sql_query("SELECT * FROM noticias", connection)

print("noticias:", noticias.shape)
print(noticias.columns.tolist())

C:\Users\Usuario\AppData\Local\Temp\ipykernel_20788\2652879179.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  noticias = pd.read_sql_query("SELECT * FROM noticias", connection)


noticias: (10, 2)
['cuerpo', 'titular']


In [18]:
# Inspeccionar datos: 
noticias.head()

,cuerpo,titular
0,fue rodríguez zapatero el que empezó a utiliza...,escenas de geometría variable
1,la reunión que mantendrán este viernes el dipu...,iglesias se reunirá con junqueras sin cristal ...
2,"bankia abona este jueves, 11 de abril, su quin...",bankia paga otros 219 millones al estado vía d...
3,independientemente de los pactos a los que lle...,sánchez tiene ya una mayoría en el congreso pa...
4,"de los amigos me guarde dios, que de mis enemi...",un presidente irresponsable


In [83]:
# Leer el .env ubicado junto al notebook, no uno relativo al directorio del kernel
import os
from pathlib import Path
from dotenv import dotenv_values

directorio_actual = Path.cwd().resolve()
directorio_proyecto = next(
    (ruta for ruta in (directorio_actual, *directorio_actual.parents)
     if (ruta / "edgardo-cerda.ipynb").is_file()),
    None,
)
if directorio_proyecto is None:
    raise FileNotFoundError(
        f"No se encontró edgardo-cerda.ipynb desde el directorio {directorio_actual}"
    )

ruta_env = directorio_proyecto / ".env"
if not ruta_env.is_file():
    raise FileNotFoundError(f"No se encontró el archivo de configuración: {ruta_env}")

OPENROUTER_KEY = dotenv_values(ruta_env).get("OPENROUTER_API_KEY")
if not OPENROUTER_KEY:
    raise ValueError(f"No se encontró OPENROUTER_API_KEY en {ruta_env}")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_KEY
print(f"Archivo .env cargado: {ruta_env}")

Archivo .env cargado: C:\Users\Usuario\Documents\personal\busqueda laboral\pruebas tecnicas\INE-Ciencia de datos\prueba-tecnica\.env


In [80]:
MODELO = os.getenv("MODELO_LLM", "google/gemini-2.5-flash-lite")  
MENSAJE_ESTANDAR = (
    "Eres un analista de noticias. Lee la siguiente noticia y entrega un "
    "resumen de máximo 3 líneas.\n\n"
    "Titular: {titular}\n\n"
    "Cuerpo:\n{cuerpo}"
)

COLUMNAS = ["titular", "cuerpo"]
URL = "https://openrouter.ai/api/v1/chat/completions"


def consultar_openrouter(prompt: str, modelo: str = MODELO, temperatura: float = 0.2,
                         reintentos: int = 3, timeout: int = 60) -> str:
    """Envía un prompt a OpenRouter y devuelve el texto de la respuesta."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": modelo,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperatura,
    }

    for intento in range(1, reintentos + 1):
        try:
            resp = requests.post(URL, headers=headers, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except (requests.RequestException, KeyError, IndexError) as e:
            if intento == reintentos:
                return f"ERROR: {e}"
            time.sleep(2 ** intento)  # espera exponencial antes de reintentar


def procesar_noticias(noticia: pd.DataFrame, columnas: list = COLUMNAS,
                      plantilla: str = MENSAJE_ESTANDAR, pausa: float = 0.5) -> pd.DataFrame:
    """Consulta OpenRouter por cada fila usando titular y cuerpo; agrega la columna 'respuesta'."""
    faltantes = [c for c in columnas if c not in noticia.columns]
    if faltantes:
        raise KeyError(f"Faltan columnas en el DataFrame: {faltantes}")

    respuestas = []
    for i, fila in enumerate(noticia[columnas].fillna("").itertuples(index=False), start=1):
        prompt = plantilla.format(**fila._asdict())   # rellena {titular} y {cuerpo}
        respuestas.append(consultar_openrouter(prompt))
        print(f"Procesada {i}/{len(noticia)}")
        time.sleep(pausa)  # evita saturar la API

    resultado = noticia.copy()
    resultado["respuesta"] = respuestas
    return resultado

In [81]:
resultado = procesar_noticias(noticias)

Procesada 1/10
Procesada 2/10
Procesada 3/10
Procesada 4/10
Procesada 5/10
Procesada 6/10
Procesada 7/10
Procesada 8/10
Procesada 9/10
Procesada 10/10


In [82]:
resultado

,cuerpo,titular,respuesta
0,fue rodríguez zapatero el que empezó a utiliza...,escenas de geometría variable,"El artículo compara la estrategia de ""geometrí..."
1,la reunión que mantendrán este viernes el dipu...,iglesias se reunirá con junqueras sin cristal ...,Pablo Iglesias se reunirá con Oriol Junqueras ...
2,"bankia abona este jueves, 11 de abril, su quin...",bankia paga otros 219 millones al estado vía d...,Bankia pagará 219 millones de euros al Estado ...
3,independientemente de los pactos a los que lle...,sánchez tiene ya una mayoría en el congreso pa...,El nuevo Congreso otorga a las fuerzas de izqu...
4,"de los amigos me guarde dios, que de mis enemi...",un presidente irresponsable,El rey emérito Juan Carlos I ha regularizado c...
5,"el juez del caso nóos, josé castro, ha ordenad...",castro embarga el palacete y otra finca a mata...,El juez José Castro ha ordenado el embargo de ...
6,"el presidente del gobierno y del pp, mariano r...","el jefe del ejecutivo, dispuesto a reunirse co...",Mariano Rajoy se muestra abierto a reunirse co...
7,el juicio al procés en el tribunal supremo cas...,el último choque con puigdemont y las diferenc...,"El juicio del ""procés"" se acerca, con fecha pr..."
8,"solo tiene 42 años, era un policía de raza», q...",interior jubila al policía que perdió la visió...,Un policía de 42 años ha sido jubilado forzosa...
9,"el coordinador general de eh bildu, arnaldo ot...","otegi sobre bateragune ""ni nos domaron, ni nos...",El Tribunal Supremo reabrirá el juicio contra ...


# 4. Cuéntanos sobre un proyecto

Mi experiencia más directa es el desarrollo de un modelo de codificación automático utilizando redes neuronales, para la codificación de las glosas abiertas de Censo de ocupación principal (CIUO-08.cl, a 1 dígito) y de actividad económica (CAENES, a 1 dígito). Las prediccines de este modelo fueron utilizadas en los resultados oficiales de Censo, sobre más de 14 millones de ocupadas y ocupadas del país.

El problama que se intentaba resolver era el de codificar las glosas abiertas de censo, considerando que el alto número de respuestas y su diversidad léxica (que implicaba más de 7 millones de respuestas únicas), dificultaba su codificación manual o por reglas simples, haciendo que esto fuera fuera excesivamente costoso.

La estrategia utilizada entonces fue el desarrollo de un modelo de codificación automática que, de forma eficiente y estable, pudiera realizar estas predicciones. Para esto se comenzó con la generación de una base de entrenamiento a partir de la codificación manual de glosas de Censo, para que el modelo se adaptara bien a la idiosincracia y las particularidades de las glosas de censo (glosas, en general, de baja calidad, comparado con las de otros productos del INE). En esta etapa se codificaron manualmente más de 400.000 glosas únicas, inicialmente seleccionadas aleatoriamente de forma estratificada a partir de las respuestas de censo, y luego con muestreo activo (priorizando casos con peor rendimiento en una versión inicial del modelo). Luego, con estos datos se entrenó un modelo de redes neuronales que utilizaba BETO para generar embeddings, y una capa MLP con 3 capas. Adicionalmente, se utilizaron técnicas de data-augmentation (casos con error ortográfico introducido y casos con correcciones ortográficas), además de dropout durante el entrenamiento para mejorar su generalización. El proceso fue supervisado y validado por el equipo de Nomenclaturas del INE, experto en codificación. El resultado fue un modelo con buenas métricas de calidad.

En términos de aprendizaje, diría que, si se va a contar con codificación manual, es importante planificar con anticipación qué es lo que se quiere lograr. En nuestro caso se codificó manualmente a 1 dígito, pero se pudo haber sido más ambicioso y codificar a 4. Una vez tomada esa decisión inicial ya resultaba difícil revertirlo. Los requerimientos técnicos también son claves, ya que el modelo se entrenó y predijo sin GPU, lo que incrementó considerablemente el tiempo necesario para el entrenamiento y el testeo. De manera inversa, así como se puede ser más ambicioso en términos del objetivo del modelo, se pudo haber utilizado técnicas y modelos más modernos que redujeran la necesidad de entrenar un modelo casi desde cero.